<a href="https://colab.research.google.com/github/Ayu-sshhhhh/NLP-by-me/blob/main/Intro_to_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to NLP Fundamentals
NLP has the goal of deriving info. from natural language (could be texts or sequence).
Another common term for NLP problems is sequence to sequence problems (seq2seq).

# Getting some pre-defined helper functions


In [1]:
!wget https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/refs/heads/main/extras/helper_functions.py

from helper_functions import unzip_data, create_tensorboard_callback, plot_loss_curves, compare_historys

--2026-07-01 05:00:41--  https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/refs/heads/main/extras/helper_functions.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10246 (10K) [text/plain]
Saving to: ‘helper_functions.py’

helper_functions.py 100%[===================>]  10.01K  --.-KB/s    in 0s      

2026-07-01 05:00:41 (68.5 MB/s) - ‘helper_functions.py’ saved [10246/10246]



# Get a text dataset
The dataset we're going to be using is Kaggle's introduction to NLP
See the original source here : https://www.kaggle.com/c/nlp-getting-started

In [60]:
import pandas as pd
from pandas import read_csv
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

### Glimpse of Data

In [61]:
train_df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [62]:
train_df["text"][0]

'Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all'

In [63]:
test_df.tail()

,id,keyword,location,text
3258,10861,NaN,NaN,EARTHQUAKE SAFETY LOS ANGELES ÛÒ SAFETY FASTE...
3259,10865,NaN,NaN,Storm in RI worse than last hurricane. My city...
3260,10868,NaN,NaN,Green Line derailment in Chicago http://t.co/U...
3261,10874,NaN,NaN,MEG issues Hazardous Weather Outlook (HWO) htt...
3262,10875,NaN,NaN,#CityofCalgary has activated its Municipal Eme...


In [64]:
len(test_df), len(train_df)

(3263, 7613)

In [65]:
# Shuffling the data (just for fun)
train_df_shuffled = train_df.sample(frac=1, random_state=42)
train_df_shuffled.head()

,id,keyword,location,text,target
2644,3796,destruction,NaN,So you have a new weapon that can cause un-ima...,1
2227,3185,deluge,NaN,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,7769,police,UK,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,191,aftershock,NaN,Aftershock back to school kick off was great. ...,0
6845,9810,trauma,"Montgomery County, MD",in response to trauma Children of Addicts deve...,0


# Split data into train and validation sets

In [66]:
from sklearn.model_selection import train_test_split

In [67]:
train_sentences, val_sentences, train_labels, val_labels = train_test_split(train_df_shuffled["text"].to_numpy(),
                                                                            train_df_shuffled["target"].to_numpy(),
                                                                            test_size=0.1,
                                                                            random_state=42)


In [68]:
len(train_sentences)

6851

In [69]:
len(val_sentences)

762

In [70]:
train_sentences[:10], train_labels[:10]

(array(['@mogacola @zamtriossu i screamed after hitting tweet',
        'Imagine getting flattened by Kurt Zouma',
        '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
        "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
        'Somehow find you and I collide http://t.co/Ee8RpOahPk',
        '@EvaHanderek @MarleyKnysh great times until the bus driver held us hostage in the mall parking lot lmfao',
        'destroy the free fandom honestly',
        'Weapons stolen from National Guard Armory in New Albany still missing #Gunsense http://t.co/lKNU8902JE',
        '@wfaaweather Pete when will the heat wave pass? Is it really going to be mid month? Frisco Boy Scouts have a canoe trip in Okla.',
        'Patient-reported outcomes in long-term survivors of metastatic colorectal cancer - British Journal of Surgery http://t.co/5Yl4DC1Tqt'],
       dtype=object),
 array([0,

# Convert texts to numbers

In [71]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

# This is a sample TextVectorizzation object
text_vectorizer = TextVectorization(max_tokens=None,
                                    standardize="lower_and_strip_punctuation",
                                    split="whitespace",
                                    ngrams=None,
                                    output_mode="int",
                                    output_sequence_length=None)

In [72]:
max_vocab_length=10000
max_length = 15

text_vectorizer = TextVectorization(max_tokens=max_vocab_length,
                                    output_mode="int",
                                    output_sequence_length=max_length)

In [73]:
text_vectorizer.adapt(train_sentences)

In [74]:
sample_sentence = "My name is Ana, I am from England. Nice to meet you !"
text_vectorizer([sample_sentence])

<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[  13,  735,    9,    1,    8,  160,   20, 2126, 1117,    5, 1389,
          12,    0,    0,    0]])>

### Tokenizing a random sentence

In [75]:
import random
random_sentence = random.choice(train_sentences)
print(f"Original Sentence : \n{random_sentence}\
\n\nVectorized version :")
text_vectorizer([random_sentence])

Original Sentence : 
Car engulfed in flames backs up traffic at ParleyÛªs Summit http://t.co/RmucfjCaZr

Vectorized version :


<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[ 127,  436,    4,  218, 6145,   27,  592,   17, 9941, 4503,    1,
           0,    0,    0,    0]])>

### Unique words in vocab

Here, [UNK] token is used for 'unknown' words

In [76]:
words_in_vocab = text_vectorizer.get_vocabulary()
top_5_words = words_in_vocab[:5]
bottom_5_words = words_in_vocab[-5:]
print(f"Number of words in vocab : {len(words_in_vocab)}")
print(f"Top 5 words in vocab : {top_5_words}")
print(f"Bottom 5 words in vocab : {bottom_5_words}")

Number of words in vocab : 10000
Top 5 words in vocab : ['', '[UNK]', np.str_('the'), np.str_('a'), np.str_('in')]
Bottom 5 words in vocab : [np.str_('pages'), np.str_('paeds'), np.str_('pads'), np.str_('padres'), np.str_('paddytomlinson1')]


In [77]:
words_in_vocab[12:15]

[np.str_('you'), np.str_('my'), np.str_('with')]

# Cooking an Embedding layer

In [78]:
tf.random.set_seed(42)
from tensorflow.keras import layers
embedding = layers.Embedding(input_dim=max_vocab_length,
                             output_dim = 128,
                             embeddings_initializer="uniform",
                             name="embedding_1")
embedding

<Embedding name=embedding_1, built=False>

As seen above, `built=False` means :
Keras uses a *lazy initialization* strategy. When you define layers.Embedding(...), Keras registers the configuration you want (like the vocabulary size and output dimensions), but it does not actually create the weight matrices in memory yet.

It waits to officially "build" the layer and allocate memory until one of two things happens:

* You pass some actual data through the layer for the first time.
* You manually call the .build(input_shape) method.

In [79]:
sample_embed = embedding(text_vectorizer(["Hehe, haha, huhu"]))
sample_embed

<tf.Tensor: shape=(1, 15, 128), dtype=float32, numpy=
array([[[ 0.0446152 , -0.04271768, -0.01799991, ..., -0.04359353,
         -0.00904464,  0.04487154],
        [ 0.04989709,  0.01524783, -0.04332993, ...,  0.04299774,
         -0.03609561,  0.00194261],
        [ 0.0446152 , -0.04271768, -0.01799991, ..., -0.04359353,
         -0.00904464,  0.04487154],
        ...,
        [ 0.01134516, -0.04477748,  0.0247211 , ...,  0.01600057,
         -0.02400251,  0.00383974],
        [ 0.01134516, -0.04477748,  0.0247211 , ...,  0.01600057,
         -0.02400251,  0.00383974],
        [ 0.01134516, -0.04477748,  0.0247211 , ...,  0.01600057,
         -0.02400251,  0.00383974]]], dtype=float32)>

In [80]:
sample_embed[0][0]

<tf.Tensor: shape=(128,), dtype=float32, numpy=
array([ 0.0446152 , -0.04271768, -0.01799991,  0.00628506, -0.03780404,
       -0.00017043, -0.03974702,  0.03650628,  0.01196331, -0.04559486,
       -0.04255487, -0.03424229,  0.04479085, -0.00489329, -0.02331594,
       -0.04806655,  0.04886791, -0.04307793, -0.0338247 , -0.01178928,
       -0.02982954,  0.0321981 , -0.0069439 , -0.00177655,  0.0484491 ,
        0.00338789, -0.04550054, -0.00624744, -0.0304528 , -0.03021236,
       -0.03245269, -0.02678796,  0.03395278, -0.01852781,  0.02363128,
        0.04387157,  0.02606593, -0.04100634,  0.0149654 ,  0.02663883,
       -0.04660839, -0.01523408, -0.04718117, -0.02764515,  0.01864283,
        0.03842061, -0.0229589 , -0.00813484,  0.02172071,  0.00481411,
        0.01481875, -0.01609235,  0.01957396,  0.03102392, -0.03051616,
        0.00979463, -0.01048908,  0.04661155, -0.04728613, -0.00376559,
       -0.00927942,  0.01368858, -0.01303421,  0.043023  ,  0.04391081,
       -0.037419

# Modelling

* **Model 0** : Naive Bayes (baseline)
* **Model 1** : Feed-forward NN (dense model)
* **Model 2** : LSTM model
* **Model 3** : GRU model
* **Model 4** : Bidirectional-LSTM model
* **Model 5** : 1D Conv NN
* **Model 6** : TFHub pre-trained feature extractor
* **Model 7** : Same as model 6 with 10 % of training data

## Model 0

In [81]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

model_0 = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB())
])
model_0.fit(train_sentences, train_labels)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())])

In [82]:
baseline_score = model_0.score(val_sentences, val_labels)
print(f"Our baseline model achieves an accuracy of : {baseline_score*100:.2f} %")

Our baseline model achieves an accuracy of : 79.27 %


In [83]:
# Making predictions with our baseline model

baseline_preds = model_0.predict(val_sentences)
baseline_preds[:20]

array([1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1])

In [84]:
# Function to evaluate : accuracy, precision, recall, f1-score
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def calculate_results(y_true, y_pred):
  """
  Calculates model accuracy, precision, recall and f1 score of a binary classification model.

  Args:
  y_true = true labels in the form of a 1D array
  y_pred = predicted labels in the form of 1D array

  Returns a dict of accuracy, precision, recall, f1-score.
  """
  model_accuracy = accuracy_score(y_true, y_pred) *100
  model_precision, model_recall, model_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")
  model_results = {"accuracy" : model_accuracy,
                   "precision" : model_precision,
                   "recall" : model_recall,
                   "f1" : model_f1}
  return model_results

In [85]:
baseline_results = calculate_results(y_true = val_labels,
                                     y_pred = baseline_preds)
baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1': 0.7862189758049549}

## Model 1

In [86]:
from tensorflow.keras import layers
inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = embedding(x)
x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model_1 = tf.keras.Model(inputs,outputs, model="model_1_dense")

In [87]:
model_1.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

In [88]:
model_1.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_3            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,280,129 (4.88 MB)

 Trainable params: 1,280,129 (4.88 MB)

 Non-trainable params: 0 (0.00 B)

In [89]:
model_1_history = model_1.fit(train_sentences,
                              train_labels,
                              epochs=5,
                              validation_data=(val_sentences, val_labels))

Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.6919 - loss: 0.6074 - val_accuracy: 0.7612 - val_loss: 0.5350
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.8156 - loss: 0.4433 - val_accuracy: 0.7900 - val_loss: 0.4745
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.8580 - loss: 0.3508 - val_accuracy: 0.7940 - val_loss: 0.4621
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.8885 - loss: 0.2884 - val_accuracy: 0.7848 - val_loss: 0.4680
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9095 - loss: 0.2416 - val_accuracy: 0.7795 - val_loss: 0.4835


In [90]:
model_1.evaluate(val_sentences, val_labels)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7795 - loss: 0.4835 


[0.4834817051887512, 0.7795275449752808]

In [91]:
model_1_pred_probs = model_1.predict(val_sentences)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [92]:
model_1_preds = tf.squeeze(tf.round(model_1_pred_probs))
model_1_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0.], dtype=float32)>

In [93]:
model_1_results = calculate_results(y_true=val_labels,
                                  y_pred=model_1_preds)
model_1_results

{'accuracy': 77.95275590551181,
 'precision': 0.7829461349391948,
 'recall': 0.7795275590551181,
 'f1': 0.7768651797484591}

##### Another helper Func.

In [94]:
def compare_baseline_to_new_results(baseline_results, new_model_results):
  for key, value in baseline_results.items():
    print(f"Baseline {key}: {value:.2f}, New {key}: {new_model_results[key]:.2f}, Difference {new_model_results[key]-value:.2f}")
compare_baseline_to_new_results(baseline_results=baseline_results,
                                new_model_results=model_1_results)

Baseline accuracy: 79.27, New accuracy: 77.95, Difference -1.31
Baseline precision: 0.81, New precision: 0.78, Difference -0.03
Baseline recall: 0.79, New recall: 0.78, Difference -0.01
Baseline f1: 0.79, New f1: 0.78, Difference -0.01


## Model 2

In [95]:
tf.random.set_seed(42)

from tensorflow.keras import layers
model_2_embedding = layers.Embedding(input_dim = max_vocab_length,
                                    output_dim = 128,
                                    embeddings_initializer = "uniform",
                                    name="embedding_2")
inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = model_2_embedding(x)
print(x.shape)
x = layers.LSTM(64)(x)
print(x.shape)
outputs = layers.Dense(1, activation="sigmoid")(x)
model_2 = tf.keras.Model(inputs, outputs, name="model_2_LSTM")

(None, 15, 128)
(None, 64)


In [96]:
model_2.compile(loss="binary_crossentropy",
                optimizer = "Adam",
                metrics = ["accuracy"])

In [97]:
model_2.summary()

Model: "model_2_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_3            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,329,473 (5.07 MB)

 Trainable params: 1,329,473 (5.07 MB)

 Non-trainable params: 0 (0.00 B)

In [98]:
model_2_history = model_2.fit(train_sentences,
                              train_labels,
                              epochs=5,
                              validation_data=(val_sentences, val_labels))

Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 29ms/step - accuracy: 0.7447 - loss: 0.5122 - val_accuracy: 0.7782 - val_loss: 0.4577
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - accuracy: 0.8716 - loss: 0.3137 - val_accuracy: 0.7585 - val_loss: 0.5229
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.9206 - loss: 0.2149 - val_accuracy: 0.7559 - val_loss: 0.6636
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.9492 - loss: 0.1529 - val_accuracy: 0.7559 - val_loss: 0.6476
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.9577 - loss: 0.1225 - val_accuracy: 0.7822 - val_loss: 0.6572


In [99]:
# Make predictions on the validation dataset
model_2_pred_probs = model_2.predict(val_sentences)

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step


In [100]:
model_2_preds = tf.squeeze(tf.round(model_2_pred_probs))

In [101]:
# Calculate LSTM model results
model_2_results = calculate_results(y_true=val_labels,
                                    y_pred=model_2_preds)
model_2_results

{'accuracy': 78.21522309711287,
 'precision': 0.7835553063606987,
 'recall': 0.7821522309711286,
 'f1': 0.7804155135464126}

In [102]:
compare_baseline_to_new_results(baseline_results, model_2_results)

Baseline accuracy: 79.27, New accuracy: 78.22, Difference -1.05
Baseline precision: 0.81, New precision: 0.78, Difference -0.03
Baseline recall: 0.79, New recall: 0.78, Difference -0.01
Baseline f1: 0.79, New f1: 0.78, Difference -0.01


## Model 3

In [103]:
tf.random.set_seed(42)
from tensorflow.keras import layers
model_3_embedding = layers.Embedding(input_dim=max_vocab_length,
                                     output_dim=128,
                                     embeddings_initializer="uniform",
                                     input_length = max_length,
                                     name="embedding_3")

inputs= layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = model_3_embedding(x)
x = layers.GRU(64)(x)
outputs= layers.Dense(1, activation="sigmoid")(x)
model_3 = tf.keras.Model(inputs, outputs, name="model_3_GRU")


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [104]:
model_3.compile(loss="binary_crossentropy",
                optimizer="Adam",
                metrics=["accuracy"])

In [105]:
model_3.summary()

Model: "model_3_GRU"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_3            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_3 (Embedding)         │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,317,313 (5.03 MB)

 Trainable params: 1,317,313 (5.03 MB)

 Non-trainable params: 0 (0.00 B)

In [106]:
model_3_history = model_3.fit(train_sentences,
                              train_labels,
                              epochs=5,
                              validation_data = (val_sentences, val_labels))

Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - accuracy: 0.7288 - loss: 0.5246 - val_accuracy: 0.7730 - val_loss: 0.4601
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 39ms/step - accuracy: 0.8694 - loss: 0.3153 - val_accuracy: 0.7638 - val_loss: 0.5097
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - accuracy: 0.9167 - loss: 0.2199 - val_accuracy: 0.7638 - val_loss: 0.5888
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.9416 - loss: 0.1663 - val_accuracy: 0.7677 - val_loss: 0.5658
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - accuracy: 0.9556 - loss: 0.1379 - val_accuracy: 0.7598 - val_loss: 0.6459


In [107]:
model_3_pred_probs = model_3.predict(val_sentences)
model_3_preds = tf.squeeze(tf.round(model_3_pred_probs))
model_3_results = calculate_results(y_true=val_labels,
                                    y_pred=model_3_preds)
model_3_results

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step


{'accuracy': 75.98425196850394,
 'precision': 0.760844531051619,
 'recall': 0.7598425196850394,
 'f1': 0.7578634561555267}

In [108]:
compare_baseline_to_new_results(baseline_results, model_3_results)

Baseline accuracy: 79.27, New accuracy: 75.98, Difference -3.28
Baseline precision: 0.81, New precision: 0.76, Difference -0.05
Baseline recall: 0.79, New recall: 0.76, Difference -0.03
Baseline f1: 0.79, New f1: 0.76, Difference -0.03


## Model 4

In [109]:
tf.random.set_seed(42)
from tensorflow.keras import layers
model_4_embedding = layers.Embedding(input_dim=max_vocab_length,
                                     output_dim=128,
                                     embeddings_initializer="uniform",
                                     input_length = max_length,
                                     name="embedding_4")

inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = model_4_embedding(x)
x = layers.Bidirectional(layers.LSTM(64))(x)
outputs=layers.Dense(1, activation="sigmoid")(x)
model_4 = tf.keras.Model(inputs, outputs, name="model_4_bidirectional")


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [110]:
model_4.compile(loss="binary_crossentropy",
                optimizer="Adam",
                metrics=["accuracy"])

In [111]:
model_4.summary()

Model: "model_4_bidirectional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_3            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_4 (Embedding)         │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,378,945 (5.26 MB)

 Trainable params: 1,378,945 (5.26 MB)

 Non-trainable params: 0 (0.00 B)

In [112]:
model_4_history = model_4.fit(train_sentences,
                              train_labels,
                              epochs = 5,
                              validation_data=(val_sentences, val_labels))

Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 12s 34ms/step - accuracy: 0.7444 - loss: 0.5122 - val_accuracy: 0.7795 - val_loss: 0.4611
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 36ms/step - accuracy: 0.8720 - loss: 0.3118 - val_accuracy: 0.7625 - val_loss: 0.5063
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - accuracy: 0.9225 - loss: 0.2114 - val_accuracy: 0.7585 - val_loss: 0.5916
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 36ms/step - accuracy: 0.9494 - loss: 0.1482 - val_accuracy: 0.7572 - val_loss: 0.6839
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 10s 37ms/step - accuracy: 0.9622 - loss: 0.1161 - val_accuracy: 0.7467 - val_loss: 0.7119


In [113]:
model_4_pred_probs = model_4.predict(val_sentences)
model_4_preds = tf.squeeze(tf.round(model_4_pred_probs))
model_4_results = calculate_results(val_labels, model_4_preds)
model_4_results

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step


{'accuracy': 74.67191601049869,
 'precision': 0.747051341252852,
 'recall': 0.7467191601049868,
 'f1': 0.7468530612601981}

In [114]:
compare_baseline_to_new_results(baseline_results, model_4_results)

Baseline accuracy: 79.27, New accuracy: 74.67, Difference -4.59
Baseline precision: 0.81, New precision: 0.75, Difference -0.06
Baseline recall: 0.79, New recall: 0.75, Difference -0.05
Baseline f1: 0.79, New f1: 0.75, Difference -0.04


## Model 5

In [115]:
tf.random.set_seed(42)
from tensorflow.keras import layers
model_5_embedding = layers.Embedding(input_dim=max_vocab_length,
                                     output_dim = 128,
                                     embeddings_initializer="uniform",
                                     input_length=max_length,
                                     name="embedding")
from tensorflow.keras import layers
inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = model_5_embedding(x)
x = layers.Conv1D(filters=32, kernel_size=5, activation="relu")(x)
x = layers.GlobalMaxPool1D()(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model_5 = tf.keras.Model(inputs, outputs, name="model_5_Conv1D")

model_5.compile(loss="binary_crossentropy",
                optimizer="Adam",
                metrics=["accuracy"])

model_5.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "model_5_Conv1D"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_3            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 11, 32)         │        20,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 32)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,300,545 (4.96 MB)

 Trainable params: 1,300,545 (4.96 MB)

 Non-trainable params: 0 (0.00 B)

In [116]:
model_5_history = model_5.fit(train_sentences, train_labels,
                              epochs = 5,
                              validation_data=(val_sentences, val_labels))

Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.7214 - loss: 0.5647 - val_accuracy: 0.7835 - val_loss: 0.4725
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.8587 - loss: 0.3429 - val_accuracy: 0.7795 - val_loss: 0.4788
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.9225 - loss: 0.2168 - val_accuracy: 0.7756 - val_loss: 0.5361
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 6s 27ms/step - accuracy: 0.9539 - loss: 0.1414 - val_accuracy: 0.7769 - val_loss: 0.6050
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - accuracy: 0.9675 - loss: 0.0991 - val_accuracy: 0.7769 - val_loss: 0.6717


In [117]:
model_5_pred_probs = model_5.predict(val_sentences)
model_5_preds = tf.squeeze(tf.round(model_5_pred_probs))
model_5_results = calculate_results(val_labels, model_5_preds)
model_5_results

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step


{'accuracy': 77.69028871391076,
 'precision': 0.7822241302284023,
 'recall': 0.7769028871391076,
 'f1': 0.7734519762210931}

In [118]:
compare_baseline_to_new_results(baseline_results, model_5_results)

Baseline accuracy: 79.27, New accuracy: 77.69, Difference -1.57
Baseline precision: 0.81, New precision: 0.78, Difference -0.03
Baseline recall: 0.79, New recall: 0.78, Difference -0.02
Baseline f1: 0.79, New f1: 0.77, Difference -0.01


## Model 6
The main difference between the *embedding layer* we created and the Universal Sentence Encoder is that rather than create a `word-level embedding`, the *Universal Sentence Encoder* creates a whole `sentence-level embedding`.

In [119]:
import tensorflow_hub as hub
embed = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")
embed_samples = embed([sample_sentence,
                       "The cost of loving someone is never loving anyone"])
print(embed_samples[0][:50])

tf.Tensor(
[ 0.03438964 -0.0693329  -0.01568206  0.03749776  0.04257107 -0.04355963
 -0.02749122 -0.06338162  0.06438294 -0.06808515  0.04316942 -0.06025265
  0.03982021  0.05711937 -0.0038758  -0.08979888 -0.02786009  0.01527802
 -0.05669393 -0.06490707 -0.04948111  0.05137116  0.07542051  0.03166074
  0.01216986  0.00166555  0.01277624  0.02627511 -0.05267827 -0.04881942
 -0.06625039  0.01631885  0.0480864   0.02760794  0.00706309  0.05434644
  0.01578761 -0.0257317   0.00311233  0.07882057  0.0097094  -0.02285167
 -0.00590073  0.02730261  0.0549678   0.05044248 -0.09082599 -0.00563833
 -0.03992612 -0.02984056], shape=(50,), dtype=float32)


In [120]:
embed_samples.shape

TensorShape([2, 512])

In [121]:
sentence_encoder_layer = hub.KerasLayer("https://tfhub.dev/google/universal-sentence-encoder/4",
                                        input_shape=[],
                                        dtype=tf.string,
                                        trainable=False,
                                        name="USE")

In [137]:
import tf_keras as tk
model_6 = tk.Sequential([
    sentence_encoder_layer,
    tk.layers.Dense(64, activation="relu"),
    tk.layers.Dense(1, activation="sigmoid")
], name="model_6_USE")

model_6.compile(loss="binary_crossentropy",
                optimizer="Adam",
                metrics=["accuracy"])

model_6.summary()

Model: "model_6_USE"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 USE (KerasLayer)            (None, 512)               256797824 
                                                                 
 dense (Dense)               (None, 64)                32832     
                                                                 
 dense_1 (Dense)             (None, 1)                 65        
                                                                 
Total params: 256830721 (979.73 MB)
Trainable params: 32897 (128.50 KB)
Non-trainable params: 256797824 (979.61 MB)
_________________________________________________________________


In [139]:
model_6_history = model_6.fit(train_sentences, train_labels,
                              epochs = 5,
                              validation_data = (val_sentences, val_labels))

Epoch 1/5
215/215 [==============================] - 13s 34ms/step - loss: 0.4989 - accuracy: 0.7895 - val_loss: 0.4493 - val_accuracy: 0.8045
Epoch 2/5
215/215 [==============================] - 3s 14ms/step - loss: 0.4140 - accuracy: 0.8149 - val_loss: 0.4387 - val_accuracy: 0.8123
Epoch 3/5
215/215 [==============================] - 3s 13ms/step - loss: 0.3995 - accuracy: 0.8225 - val_loss: 0.4346 - val_accuracy: 0.8136
Epoch 4/5
215/215 [==============================] - 5s 23ms/step - loss: 0.3922 - accuracy: 0.8266 - val_loss: 0.4300 - val_accuracy: 0.8097
Epoch 5/5
215/215 [==============================] - 4s 21ms/step - loss: 0.3858 - accuracy: 0.8295 - val_loss: 0.4317 - val_accuracy: 0.8163


In [140]:
model_6_pred_probs = model_6.predict(val_sentences)
model_6_preds = tf.squeeze(tf.round(model_6_pred_probs))
model_6_results = calculate_results(val_labels, model_6_preds)
model_6_results

24/24 [==============================] - 1s 17ms/step


{'accuracy': 81.62729658792651,
 'precision': 0.8187546712946793,
 'recall': 0.8162729658792651,
 'f1': 0.8147089025083661}

In [141]:
compare_baseline_to_new_results(baseline_results, model_6_results)

Baseline accuracy: 79.27, New accuracy: 81.63, Difference 2.36
Baseline precision: 0.81, New precision: 0.82, Difference 0.01
Baseline recall: 0.79, New recall: 0.82, Difference 0.02
Baseline f1: 0.79, New f1: 0.81, Difference 0.03
